dynamic_agnet_factory
=================
# agent_factory/dynamic_agent_factory.py
import asyncio
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, Tool
from mcp_clients.weather_client import get_weather
from mcp_clients.pollution_client import get_pollution
from mcp_clients.math_client import add, multiply

class DynamicAgentFactory:
    def __init__(self, agent_config):
        self.agent_config = agent_config
        self._agents = {}

    async def get_agent(self, name: str):
        """Create or fetch an existing agent dynamically."""
        if name in self._agents:
            return self._agents[name]

        if name not in self.agent_config:
            raise ValueError(f"Agent '{name}' not found in configuration.")

        config = self.agent_config[name]

        # Initialize LLM
        if config["llm"] == "openai":
            llm = ChatOpenAI(
                model=config["llm_model"],
                temperature=0.3,
                api_key=os.getenv("OPENAI_API_KEY")
            )
        else:
            raise ValueError(f"Unsupported LLM type: {config['llm']}")

        # Create toolset dynamically
        tools = []
        for mcp_name in config.get("mcp_servers", []):
            if mcp_name == "math-mcp":
                tools.append(Tool(name="add", func=add, description="Add two numbers"))
                tools.append(Tool(name="multiply", func=multiply, description="Multiply two numbers"))
            elif mcp_name == "weather-mcp":
                tools.append(Tool(name="get_weather", func=get_weather, description="Get real-time weather info"))
            elif mcp_name == "pollution-mcp":
                tools.append(Tool(name="get_pollution", func=get_pollution, description="Fetch pollution levels"))

        # Initialize LangChain agent
        agent = initialize_agent(
            tools=tools,
            llm=llm,
            agent="zero-shot-react-description",
            verbose=True,
            system_message=config.get("system_prompt", "You are a helpful assistant.")
        )

        self._agents[name] = agent
        return agent


Excellent 🔥 — you’ve structured your **math MCP client** beautifully with self-contained parsing logic and a clean `get_tools()` function.

Let’s now do the same for **`pollution-mcp_client.py`** — with multiple realistic tools for air quality, emissions, and city ranking.

Below is a **complete and ready-to-integrate** version for your pollution MCP client.

---

## 🌍 `mcp_clients/pollution-mcp_client.py`

```python
# mcp_clients/pollution-mcp_client.py
from langchain.agents import Tool
import random

# --- Core data and logic ---

POLLUTION_DATA = {
    "Delhi": {"PM2.5": 185, "AQI": "Poor", "CO2": 410, "NO2": 45},
    "Bangalore": {"PM2.5": 78, "AQI": "Moderate", "CO2": 390, "NO2": 30},
    "Mumbai": {"PM2.5": 122, "AQI": "Unhealthy", "CO2": 400, "NO2": 38},
    "Tokyo": {"PM2.5": 42, "AQI": "Good", "CO2": 370, "NO2": 22},
    "New York": {"PM2.5": 60, "AQI": "Moderate", "CO2": 380, "NO2": 28},
}


def get_pollution(city: str) -> str:
    """Get pollution data for a city."""
    data = POLLUTION_DATA.get(city.title())
    if data:
        return (
            f"Pollution in {city.title()} — PM2.5: {data['PM2.5']} µg/m³, "
            f"AQI: {data['AQI']}, CO₂: {data['CO2']} ppm, NO₂: {data['NO2']} µg/m³"
        )
    else:
        return f"No pollution data found for {city}."


def compare_pollution(city1: str, city2: str) -> str:
    """Compare pollution between two cities."""
    d1, d2 = POLLUTION_DATA.get(city1.title()), POLLUTION_DATA.get(city2.title())
    if not d1 or not d2:
        return f"Data missing for one of the cities: {city1}, {city2}."
    worse = city1 if d1["PM2.5"] > d2["PM2.5"] else city2
    return (
        f"{worse.title()} has higher PM2.5 levels — "
        f"{POLLUTION_DATA[worse.title()]['PM2.5']} µg/m³ vs "
        f"{POLLUTION_DATA[city1.title()]['PM2.5']} µg/m³ in {city1.title()} and "
        f"{POLLUTION_DATA[city2.title()]['PM2.5']} µg/m³ in {city2.title()}."
    )


def get_top_polluted_cities(n: int = 3) -> str:
    """Return top N polluted cities by PM2.5."""
    ranked = sorted(POLLUTION_DATA.items(), key=lambda x: x[1]["PM2.5"], reverse=True)
    top_cities = ranked[:n]
    return "\n".join([f"{city}: PM2.5 {data['PM2.5']} µg/m³ ({data['AQI']})" for city, data in top_cities])


def estimate_emissions(city: str, vehicles: int) -> str:
    """Estimate CO₂ emissions from vehicles."""
    base = POLLUTION_DATA.get(city.title(), {"CO2": 400})["CO2"]
    estimate = base + (vehicles * random.uniform(0.01, 0.05))
    return f"Estimated CO₂ level in {city.title()} with {vehicles} vehicles: {estimate:.2f} ppm"


# --- Helper parser functions ---

def _parse_and_get_pollution(query: str) -> str:
    import re
    match = re.search(r"pollution in ([A-Za-z\s]+)", query)
    if match:
        return get_pollution(match.group(1).strip())
    return "Please specify a city. Example: 'Pollution in Delhi'."


def _parse_and_compare(query: str) -> str:
    import re
    cities = re.findall(r"[A-Za-z]+", query)
    if len(cities) >= 2:
        return compare_pollution(cities[0], cities[1])
    return "Please specify two cities. Example: 'Compare Delhi and Bangalore pollution'."


def _parse_and_top(query: str) -> str:
    import re
    n = re.search(r"top (\d+)", query)
    count = int(n.group(1)) if n else 3
    return get_top_polluted_cities(count)


def _parse_and_emissions(query: str) -> str:
    import re
    match = re.search(r"(\w+)\s+(\d+)\s+vehicles", query)
    if match:
        city, num = match.group(1), int(match.group(2))
        return estimate_emissions(city, num)
    return "Please specify like: 'Estimate emissions for Delhi 2000 vehicles'."


# --- LangChain Tool registration ---

def get_tools():
    """Return a list of pollution-related tools."""
    return [
        Tool(
            name="get_pollution",
            func=_parse_and_get_pollution,
            description="Get pollution data for a city. Example: 'Pollution in Delhi'"
        ),
        Tool(
            name="compare_pollution",
            func=_parse_and_compare,
            description="Compare pollution between two cities. Example: 'Compare Delhi and Bangalore pollution'"
        ),
        Tool(
            name="get_top_polluted_cities",
            func=_parse_and_top,
            description="Get the top N polluted cities. Example: 'Top 5 polluted cities'"
        ),
        Tool(
            name="estimate_emissions",
            func=_parse_and_emissions,
            description="Estimate CO₂ emissions based on vehicles. Example: 'Estimate emissions for Delhi 2000 vehicles'"
        ),
    ]
```

---

## 🧠 Example Queries (Agent3 – pollution-mcp)

| Query                                        | Tool Used                 | Example Output                                              |
| -------------------------------------------- | ------------------------- | ----------------------------------------------------------- |
| “Pollution in Delhi”                         | `get_pollution`           | Pollution in Delhi — PM2.5: 185 µg/m³, AQI: Poor            |
| “Compare Bangalore and Mumbai pollution”     | `compare_pollution`       | Mumbai has higher PM2.5 levels...                           |
| “Top 3 polluted cities”                      | `get_top_polluted_cities` | Delhi, Mumbai, Bangalore...                                 |
| “Estimate emissions for Delhi 5000 vehicles” | `estimate_emissions`      | Estimated CO₂ level in Delhi with 5000 vehicles: 430.25 ppm |

---

✅ **Next Step Recommendation**

Now that your `pollution-mcp_client.py` supports multiple structured tools,
update your **DynamicAgentFactory** so that for `pollution-mcp`, it loads tools using:

```python
from mcp_clients.pollution-mcp_client import get_tools as get_pollution_tools
...
elif mcp_name == "pollution-mcp":
    tools.extend(get_pollution_tools())
```

Would you like me to show the **updated factory file** that automatically loads `get_tools()` for each MCP client (math, weather, pollution)?
